In [ ]:
!pip install -q groq gradio "mcp>=1.9,<2"

import os, sys, json, asyncio, textwrap
from getpass import getpass
from groq import Groq
import gradio as gr
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# ================= API key =================
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass("Paste your Groq API key: ")

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL = "openai/gpt-oss-120b"   # reliable tool calling on Groq (alt: "openai/gpt-oss-120b")

# ================= 1. MCP SERVER (written to a file) =================
SERVER_CODE = textwrap.dedent('''
    import json
    from mcp.server.fastmcp import FastMCP

    mcp = FastMCP("nutrition-server")

    # Sample nutrition data (per 100 g): calories (kcal), protein (g)
    NUTRITION_DB = {
        "chicken breast": {"calories": 165, "protein": 31.0},
        "banana":         {"calories": 89,  "protein": 1.1},
        "apple":          {"calories": 52,  "protein": 0.3},
        "egg":            {"calories": 143, "protein": 12.6},
        "rice":           {"calories": 130, "protein": 2.7},
        "chicken biryani":{"calories": 180, "protein": 8.0},
        "idli":           {"calories": 130, "protein": 4.0},
        "peanut butter":  {"calories": 588, "protein": 25.0},
        "paneer":         {"calories": 265, "protein": 18.3},
        "oats":           {"calories": 389, "protein": 16.9},
        "milk":           {"calories": 61,  "protein": 3.2},
        "almonds":        {"calories": 579, "protein": 21.2},
    }

    def _lookup(food):
        key = food.strip().lower()
        if key in NUTRITION_DB:
            return key
        for k in NUTRITION_DB:            # simple fuzzy match
            if k in key or key in k:
                return k
        return None

    @mcp.tool()
    def get_calories(food: str, grams: float = 100) -> str:
        """Return calorie information (kcal) for a food item.

        Args:
            food: Name of the food, e.g. "chicken breast"
            grams: Serving size in grams (default 100)
        """
        k = _lookup(food)
        if not k:
            return json.dumps({"error": f"'{food}' not found", "available": list(NUTRITION_DB)})
        kcal = round(NUTRITION_DB[k]["calories"] * grams / 100, 1)
        return json.dumps({"food": k, "grams": grams, "calories_kcal": kcal})

    @mcp.tool()
    def get_protein(food: str, grams: float = 100) -> str:
        """Return protein information (g) for a food item.

        Args:
            food: Name of the food, e.g. "egg"
            grams: Serving size in grams (default 100)
        """
        k = _lookup(food)
        if not k:
            return json.dumps({"error": f"'{food}' not found", "available": list(NUTRITION_DB)})
        p = round(NUTRITION_DB[k]["protein"] * grams / 100, 1)
        return json.dumps({"food": k, "grams": grams, "protein_g": p})

    if __name__ == "__main__":
        mcp.run(transport="stdio")
''')

with open("nutrition_server.py", "w") as f:
    f.write(SERVER_CODE)

SERVER_PARAMS = StdioServerParameters(command=sys.executable, args=["nutrition_server.py"])

# Colab's sys.stderr has no fileno(), so give the server a real log file
ERRLOG = open("mcp_server_stderr.log", "w")

# ================= 2. MCP CLIENT + NUTRITION AGENT =================
SYSTEM_PROMPT = """You are an expert nutritionist agent.
You have MCP tools: get_calories and get_protein. ALWAYS use them for calorie or protein
questions about a food instead of guessing. Pass the grams the user mentions (default 100).
After receiving tool results, answer clearly and briefly, stating the values returned.
If a tool says the food is not found, say so and give a rough estimate labelled as an estimate."""


def mcp_tools_to_groq(tools):
    """Convert MCP tool definitions to Groq/OpenAI function-calling format."""
    return [{
        "type": "function",
        "function": {
            "name": t.name,
            "description": t.description or "",
            "parameters": t.inputSchema,
        },
    } for t in tools]


def unwrap_errors(e):
    """Dig the real error messages out of an ExceptionGroup."""
    msgs = []
    def collect(ex):
        if isinstance(ex, BaseExceptionGroup):
            for s in ex.exceptions:
                collect(s)
        else:
            msgs.append(f"{type(ex).__name__}: {ex}")
    collect(e)
    return " | ".join(msgs)


async def _run_agent_inner(question: str):
    log = []
    async with stdio_client(SERVER_PARAMS, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            log.append("✅ Connected to MCP server: nutrition-server")

            tools = (await session.list_tools()).tools
            log.append("🔎 Discovered tools: " + ", ".join(t.name for t in tools))
            groq_tools = mcp_tools_to_groq(tools)

            messages = [{"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": question}]

            for _ in range(5):  # max tool-calling rounds
                resp = await asyncio.to_thread(
                    groq_client.chat.completions.create,
                    model=MODEL, messages=messages,
                    tools=groq_tools, tool_choice="auto", temperature=0.2,
                )
                msg = resp.choices[0].message

                if not msg.tool_calls:
                    return msg.content, log

                messages.append({
                    "role": "assistant",
                    "content": msg.content or "",
                    "tool_calls": [{
                        "id": tc.id, "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    } for tc in msg.tool_calls],
                })

                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments or "{}")
                    log.append(f"🛠️ Agent calls `{tc.function.name}` with {args}")
                    result = await session.call_tool(tc.function.name, args)
                    text = "".join(c.text for c in result.content if hasattr(c, "text"))
                    log.append(f"📦 MCP server returned: {text}")
                    messages.append({"role": "tool", "tool_call_id": tc.id, "content": text})

            return "Sorry, too many tool-call rounds.", log


async def run_agent(question: str):
    """Wrapper: shows the real error instead of a generic ExceptionGroup."""
    try:
        return await _run_agent_inner(question)
    except BaseException as e:
        raise RuntimeError(unwrap_errors(e)) from None


# ================= 3. QUICK TEST (take screenshots of this output) =================
async def demo():
    for q in ["How many calories are in 100g of chicken breast?",
              "How much protein is in 3 eggs (150 g)?"]:
        print("USER :", q)
        try:
            answer, log = await run_agent(q)
        except Exception as e:
            print("❌ ERROR:", e)
            ERRLOG.flush()
            print("--- server stderr log ---")
            print(open("mcp_server_stderr.log").read() or "(empty)")
            return
        print("\n".join(log))
        print("AGENT:", answer, "\n" + "-" * 60)

await demo()

# ================= 4. GRADIO UI =================
async def ui_ask(question):
    if not question or not question.strip():
        return "Please type a question.", ""
    try:
        answer, log = await run_agent(question.strip())
    except Exception as e:
        return f"Error: `{e}`", ""
    return answer, "\n\n".join(log)


with gr.Blocks(theme=gr.themes.Soft(primary_hue="green"), title="Nutrition MCP Agent") as demo_ui:
    gr.Markdown("# 🥗 Nutrition AI Agent + MCP\nThe agent discovers and calls tools on an MCP server "
                "(`get_calories`, `get_protein`).")
    q = gr.Textbox(label="Ask a question",
                   placeholder="e.g. How many calories are in 100g of chicken breast?")
    btn = gr.Button("Ask 🔍", variant="primary")
    ans = gr.Markdown(label="Agent answer")
    trace = gr.Markdown(label="MCP connection & tool-call log")
    gr.Examples([["How many calories are in 100g of chicken breast?"],
                 ["How much protein is in 200g of paneer?"],
                 ["Calories and protein in 2 bananas (240 g)?"]], inputs=[q])
    btn.click(ui_ask, q, [ans, trace])
    q.submit(ui_ask, q, [ans, trace])

demo_ui.launch(share=True, debug=True)

Paste your Groq API key: ··········
USER : How many calories are in 100g of chicken breast?
✅ Connected to MCP server: nutrition-server
🔎 Discovered tools: get_calories, get_protein
🛠️ Agent calls `get_calories` with {'food': 'chicken breast', 'grams': 100}
📦 MCP server returned: {"food": "chicken breast", "grams": 100.0, "calories_kcal": 165.0}
AGENT: 100 g of chicken breast contains **165 kcal**. 
------------------------------------------------------------
USER : How much protein is in 3 eggs (150 g)?
✅ Connected to MCP server: nutrition-server
🔎 Discovered tools: get_calories, get_protein
🛠️ Agent calls `get_protein` with {'food': 'egg', 'grams': 150}
📦 MCP server returned: {"food": "egg", "grams": 150.0, "protein_g": 18.9}
AGENT: Three eggs (≈150 g) contain about **18.9 g of protein**. 
------------------------------------------------------------


/tmp/ipykernel_12808/824153646.py:207: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="green"), title="Nutrition MCP Agent") as demo_ui:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://764fdead7d4823c345.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
